# 04 · Build searchable passages (no BigQuery, no quota)

Turns the text saved by notebook 03 into labelled passages, using the tested code in `src/patentsearch/ingest/patents.py`:

- **Abstract**
- **Each independent claim** (very long claims split into parts)
- **Dependent claims**, grouped to about 200 words ("Claims 2-5")

It also prepares each test case's **query** (its first independent claim, the way a searcher would start) and the **relevance labels** for the study: which corpus patents each test case cites, and whether the **examiner**, the **applicant** or **both** cited them, plus whether the cited patent has the **same applicant** as the test case.

Saves `patents.parquet`, `passages.parquet`, `testset.parquet` and `qrels.parquet` to Drive. A CPU runtime is enough; it takes about a minute.

In [ ]:
BRANCH = "v2-corpus"
!rm -rf /content/app
!git clone -q --depth 1 -b {BRANCH} https://github.com/Renuk9390/patent-prior-art-explorer.git /content/app
import sys; sys.path.insert(0, "/content/app/src")

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/patent-search"

import pandas as pd
from patentsearch.ingest.patents import build_passages, first_independent_claim, split_claims
from patentsearch.eval.qrels import citation_source, same_applicant

corpus = pd.read_parquet(f"{DRIVE_DIR}/corpus_text.parquet")
tests = pd.read_parquet(f"{DRIVE_DIR}/testset_text.parquet")
print(f"{len(corpus):,} corpus patents, {len(tests):,} test cases")

## 1 · Check the claim parser on real data

In [ ]:
import numpy as np
parsed = corpus["claims"].apply(split_claims)
n_claims = parsed.str.len()
n_indep = parsed.apply(lambda cs: sum(c.independent for c in cs))

print("Claims per patent:            median", int(n_claims.median()), "| 10th-90th pct", np.percentile(n_claims, [10, 90]).astype(int))
print("Independent claims per patent: median", int(n_indep.median()), "| 10th-90th pct", np.percentile(n_indep, [10, 90]).astype(int))
print("Patents where only 1 claim was found (possible parsing issue):", int((n_claims <= 1).sum()))
print("Patents with no independent claim found:", int((n_indep == 0).sum()))

ex = parsed.iloc[0]
print(f"\nExample {corpus.publication_number.iloc[0]}:")
for c in ex[:4]:
    print(f"  claim {c.number:2} {'INDEPENDENT' if c.independent else 'dependent  '} {c.text[:90]}...")

## 2 · Build passages

In [ ]:
rows = []
for r in corpus.itertuples():
    rows.extend(build_passages(r.publication_number, r.abstract, r.claims))
passages = pd.DataFrame(rows)

words = passages["text"].str.split().str.len()
print(f"Passages: {len(passages):,} ({len(passages) / len(corpus):.1f} per patent)")
print("By section:")
print(passages["section"].value_counts().to_string())
print("\nWords per passage: median", int(words.median()), "| 90th pct", int(np.percentile(words, 90)), "| max", int(words.max()))

print("\nExample passages:")
for p in passages[passages.publication_number == corpus.publication_number.iloc[0]].head(4).itertuples():
    print(f"  [{p.label}] {p.text[:110]}...")

## 3 · Patents table and test queries

In [ ]:
def google_patents_url(pub):
    return "https://patents.google.com/patent/" + pub.replace("-", "")

patents = corpus[["publication_number", "family_id", "title", "assignees", "filing_date",
                  "priority_date", "grant_date", "first_cpc", "g06n_codes"]].copy()
patents["url"] = patents["publication_number"].apply(google_patents_url)

tests["query"] = tests["claims"].apply(first_independent_claim)
testset = tests[["publication_number", "family_id", "title", "abstract", "assignees", "query"]].copy()
testset["url"] = testset["publication_number"].apply(google_patents_url)
missing = testset["query"].isna().sum()
testset = testset.dropna(subset=["query"])
print(f"Test cases with a usable first claim: {len(testset):,} (dropped {missing})")
print("Query length in words: median", int(testset["query"].str.split().str.len().median()))

q = testset.iloc[0]
print(f"\nExample test case {q.publication_number}: {q.title}")
print("Query (first claim):", q.query[:250], "...")
print("Applicant:", q.assignees)

## 4 · Relevance labels: who cited each corpus patent

For every test case (all publications of its family), takes every citation that points to a corpus patent and labels it **examiner**, **applicant**, **both** or **other**. Also marks whether the cited patent shares an applicant with the test case (for the portfolio-bias analysis).

In [ ]:
meta = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_meta.parquet", columns=["publication_number", "family_id"])
cit = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_citations.parquet", columns=["publication_number", "cited", "category"])
family_of = dict(zip(meta["publication_number"], meta["family_id"]))
corpus_doc_of_family = dict(zip(corpus["family_id"], corpus["publication_number"]))
query_of_family = dict(zip(testset["family_id"], testset["publication_number"]))

c = cit.assign(citing_family=cit["publication_number"].map(family_of),
               cited_family=cit["cited"].map(family_of))
c = c[c["citing_family"].isin(query_of_family) & c["cited_family"].isin(corpus_doc_of_family)]

qrels = (c.groupby(["citing_family", "cited_family"])["category"]
           .apply(citation_source).rename("source").reset_index())
qrels["query_id"] = qrels["citing_family"].map(query_of_family)
qrels["doc_id"] = qrels["cited_family"].map(corpus_doc_of_family)

q_app = dict(zip(testset["publication_number"], testset["assignees"]))
d_app = dict(zip(patents["publication_number"], patents["assignees"]))
qrels["same_applicant"] = [same_applicant(q_app[q], d_app[d]) for q, d in zip(qrels.query_id, qrels.doc_id)]
qrels = qrels[["query_id", "doc_id", "source", "same_applicant"]]

print(f"Relevance labels: {len(qrels):,} for {qrels.query_id.nunique():,} test cases "
      f"({len(qrels) / qrels.query_id.nunique():.1f} per test case)")
print("\nBy who cited it:")
print(qrels["source"].value_counts().to_string())

per_q = qrels.groupby("query_id")["source"].agg(set)
has = lambda s: per_q.apply(lambda x: s in x).sum()
print(f"\nTest cases with >=1 examiner-only citation:  {has('examiner'):,}")
print(f"Test cases with >=1 applicant-only citation: {has('applicant'):,}")
print(f"Test cases with both kinds (paired comparison): {per_q.apply(lambda x: {'examiner', 'applicant'} <= x).sum():,}")

print("\nShare of cited patents with the SAME applicant as the test case:")
print(qrels.groupby("source")["same_applicant"].mean().round(3).to_string())
print(f"all: {qrels['same_applicant'].mean():.3f}")

In [ ]:
patents.to_parquet(f"{DRIVE_DIR}/patents.parquet", index=False)
passages.to_parquet(f"{DRIVE_DIR}/passages.parquet", index=False)
testset.to_parquet(f"{DRIVE_DIR}/testset.parquet", index=False)
qrels.to_parquet(f"{DRIVE_DIR}/qrels.parquet", index=False)
import os
for f in ["patents.parquet", "passages.parquet", "testset.parquet", "qrels.parquet"]:
    print(f"{f:18} {os.path.getsize(f'{DRIVE_DIR}/{f}') / 1e6:6.1f} MB")